# Линейная регрессия и регуляризация — практическое задание

В этом ноутбуке рассматриваются методы **линейной регрессии**, **гребневой регрессии** и **Lasso-регрессии** на примере реального набора данных с Kaggle.
Ноутбук рассчитан на выполнение в **Google Colab**.


## Теоретическое введение

Линейная регрессия — это простейший алгоритм машинного обучения. Отнесение линейной регрессии к методам машинного обучения достаточно условно. На самом деле это тоже самое, что и метод наименьших квадратов.

### 1. Постановка задачи регрессии

Пусть имеется обучающая выборка
$$
\{(x_i, y_i)\}_{i=1}^n, \qquad x_i \in \mathbb{R}^d,\quad y_i \in \mathbb{R}.
$$

Здесь:
- $x_i$ — вектор признаков объекта;
- $y_i$ — числовой целевой признак.

Задача регрессии состоит в том, чтобы по признакам $x$ предсказать числовое значение $y$.

### 2. Модель линейной регрессии

Линейная регрессия предполагает, что целевая переменная приближённо выражается линейной комбинацией признаков:
$$
\hat y = w^T x + b,
$$
где:
- $w \in \mathbb{R}^d$ — вектор коэффициентов;
- $b \in \mathbb{R}$ — свободный член.

Если признаков несколько, то модель имеет вид
$$
\hat y = w_1 x_1 + w_2 x_2 + \dots + w_d x_d + b.
$$

### 3. Функция потерь

Качество модели обычно измеряется с помощью **среднеквадратичной ошибки**:
$$
\mathrm{MSE}(w,b)=\frac{1}{n}\sum_{i=1}^n (y_i-\hat y_i)^2.
$$

Минимизация этой функции приводит к подбору коэффициентов, при которых предсказания максимально близки к наблюдаемым значениям в среднем по выборке.

Также часто используют:
- **MAE**:
$$
\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^n |y_i-\hat y_i|;
$$
- **RMSE**:
$$
\mathrm{RMSE}=\sqrt{\mathrm{MSE}};
$$
- **коэффициент детерминации** $R^2$.

### 4. Коэффициент детерминации $R^2$

Коэффициент детерминации показывает, какую долю разброса целевой переменной объясняет модель.
Он вычисляется по формуле
$$
R^2 = 1 - \frac{\sum_{i=1}^n (y_i-\hat y_i)^2}{\sum_{i=1}^n (y_i-\bar y)^2},
$$
где $\bar y$ — среднее значение целевой переменной по выборке.

Смысл этой формулы таков:
- в числителе стоит сумма квадратов ошибок модели;
- в знаменателе — сумма квадратов отклонений от среднего.

Поэтому:
- $R^2 = 1$ означает идеальное предсказание;
- $R^2 = 0$ означает, что модель не лучше константного предсказания средним значением;
- $R^2 < 0$ означает, что модель работает хуже такого константного предсказания.

Важно понимать, что высокий $R^2$ не всегда гарантирует хорошее качество модели во всех смыслах: полезно одновременно анализировать и другие метрики, например MAE и RMSE.

### 5. Нормальные уравнения

Для линейной регрессии без регуляризации задача минимизации MSE допускает аналитическое решение.
Если $X$ — матрица признаков, а $y$ — вектор ответов, то при выполнении условий обратимости:
$$
\hat w = (X^T X)^{-1} X^T y.
$$

На практике для численной устойчивости часто используют более надёжные алгоритмы, реализованные в библиотеках машинного обучения.

### 6. Почему возникает необходимость регуляризации

Если признаков много, они коррелированы между собой или модель слишком гибкая, возникает риск **переобучения**:
- модель хорошо описывает обучающие данные;
- но хуже обобщает на новые объекты.

Для борьбы с этим в функцию потерь добавляют штраф за слишком большие коэффициенты модели.

### 7. Гребневая регрессия (ridge-регрессия)

Гребневая регрессия использует L2-регуляризацию:
$$
L(w,b)=\frac{1}{n}\sum_{i=1}^n (y_i-\hat y_i)^2 + \alpha \sum_{j=1}^d w_j^2.
$$

Здесь $\alpha \ge 0$ — коэффициент регуляризации.

Смысл:
- большие коэффициенты штрафуются;
- модель становится устойчивее;
- снижается дисперсия оценок.

### 8. Lasso-регрессия

Lasso использует L1-регуляризацию:
$$
L(w,b)=\frac{1}{n}\sum_{i=1}^n (y_i-\hat y_i)^2 + \alpha \sum_{j=1}^d |w_j|.
$$

Смысл:
- модель также штрафует большие коэффициенты;
- часть коэффициентов может стать в точности равной нулю;
- происходит автоматический отбор признаков.

### 9. Геометрический смысл регуляризации

- Гребневая ограничивает евклидову норму вектора коэффициентов;
- Lasso ограничивает сумму модулей коэффициентов.

Из-за различия геометрии этих ограничений Lasso чаще зануляет отдельные коэффициенты, а Ridge чаще лишь уменьшает их по модулю.

### 10. Почему важно масштабирование признаков

Регуляризованные модели чувствительны к масштабу признаков.
Если один признак измеряется в тысячах, а другой — в долях единицы, штраф на коэффициенты будет работать неравномерно.

Поэтому перед Ridge и Lasso обычно применяют стандартизацию:
$$
x_j^{\text{scaled}} = \frac{x_j - \mu_j}{\sigma_j}.
$$

### 11. Что нужно понять по итогам задания

После выполнения работы студент должен понимать:
- как строится линейная регрессия;
- как измеряется качество регрессионной модели;
- зачем нужна регуляризация;
- чем Ridge отличается от Lasso;
- как интерпретировать коэффициенты модели и результаты сравнения.


## Набор данных

Для работы используется набор данных о медицинских страховых расходах:
[https://www.kaggle.com/datasets/mirichoi0218/insurance](https://www.kaggle.com/datasets/mirichoi0218/insurance)

Целевая переменная:
- `charges` — индивидуальные медицинские расходы.

Примеры признаков:
- `age` — возраст;
- `sex` — пол;
- `bmi` — индекс массы тела;
- `children` — число детей;
- `smoker` — курит ли человек;
- `region` — регион проживания.

Этот датасет удобен для изучения линейной регрессии, потому что:
- целевая переменная является числовой;
- есть и числовые, и категориальные признаки;
- задача хорошо подходит для сравнения модели без регуляризации и с регуляризацией.

Самостоятельно выберите способ загрузить данные и сделать их доступными в блокноте с заданием.

Один из вариантов - скачивание через Kaggle API, используя свой токен. Ещё один вариант - скачать файл вручную, выложить его на Google-диск и потом подмонтировать Google-диск в среду Colab. Ниже приведены инструкции.

## Как загрузить данные с Kaggle прямо в Google Colab

Выполните предварительную растройку Kaggle и Colab (делается один раз).
1. Откройте https://www.kaggle.com, зайдите в свой профиль (создайте его, если не создали ранее), затем в настройках профиля найдите раздел "API Tokens (Recommended)"
1. Создайте токен и сразу же скопируйте его в текстовый документ на своём локальном компьютере.
1. Там же посмотрите, под каким именем пользователя (username) вы работаете в Kaggle. Скопируйте username в тот же документ
1. В Colab откройте: левая панель, вкладка "Secrets"
1. Создайте два секрета:
- `KAGGLE_USERNAME` : вставьте username
- `KAGGLE_KEY`      : вставьте token
Не забудьте установить переключатель "Доступ из блокнотов".

Далее следуйте инструкциям в коде ниже:

In [ ]:
# ШАГ 1. Установка Kaggle API
!pip -q install kaggle

# ШАГ 2. Чтение токена из Colab Secrets
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# ШАГ 3. Проверка, что всё работает
# Если всё настроено правильно — увидите список датасетов
!kaggle datasets list -s insurance | head

# ШАГ 4. Скачивание датасета
TARGET_DIR = "/content/sci-prog-and-ml" # папка для загрузки
!mkdir -p {TARGET_DIR}
os.chdir(TARGET_DIR)

# Настройка адреса скачивания: берём из URL всё, что идёт
# после datasets: (https://www.kaggle.com/datasets/mirichoi0218/insurance)
DATASET = "mirichoi0218/insurance"
!kaggle datasets download -d {DATASET} -p {TARGET_DIR} --unzip

# ШАГ 5. Проверка содержимого папки
import os
print("Содержимое папки:")
for file in os.listdir(TARGET_DIR):
    print(" -", file)

## Альтернативный вариант - скачивание файла вручную

- создайте у себя на Google-диске папку `sci-prog-and-ml`;
- скачайте файл данных вручную и поместите его в эту папку на Google-диске;
- подмонтируйте его к среде Colab, выполнив код ниже (ответьте утвердительно на запросы о предоставлении доступа);

После выполнения этих инструкций папка `sci-prog-and-ml` будет сделана рабочей и из неё уже можно будет открывать файл при помощи `df = pd.read_csv(file_name)`.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/sci-prog-and-ml')
print("Содержимое текущей папки:")
for file in os.listdir('.'):
    print(" -", file)

## Задание

### Часть 1. Загрузка и первичный анализ данных
1. Загрузите CSV-файл в `pandas.DataFrame`.
2. Выведите первые строки таблицы.
3. Определите:
   - число объектов;
   - список признаков;
   - наличие пропущенных значений;
   - типы признаков.

### Часть 2. Предобработка данных
1. Выделите целевой признак `charges`.
2. Разделите признаки на:
   - числовые;
   - категориальные.
3. Выполните кодирование категориальных признаков.
4. Подумайте, нужно ли масштабировать признаки для регуляризованных моделей.

### Часть 3. Построение моделей
1. Разделите выборку на обучающую и тестовую части.
2. Обучите:
   - обычную линейную регрессию;
   - Ridge-регрессию;
   - Lasso-регрессию.
3. Для Ridge и Lasso используйте масштабирование признаков.

### Часть 4. Оценка качества
1. Для каждой модели вычислите:
   - `MAE`,
   - `RMSE`,
   - `R^2`.
2. Сравните результаты моделей.
3. Сделайте вывод, помогла ли регуляризация.

### Часть 5. Интерпретация
1. Посмотрите на коэффициенты моделей.
2. Определите, какие признаки сильнее всего влияют на `charges`.
3. Сравните, как меняются коэффициенты при использовании регуляризации.
4. Найдите признаки, которые Lasso занулила.


## Подсказки

### Основные библиотеки
```python
import os
import glob
import numpy as np
import pandas as pd
```

### Загрузка данных

Чтобы не зависеть от точного имени файла, удобно автоматически найти CSV в рабочей папке:

```python
csv_files = glob.glob("*.csv")
print(csv_files)

df = pd.read_csv(csv_files[0])
```

### Разделение на признаки и целевую переменную
```python
X = df.drop(columns=["charges"])
y = df["charges"]
```

### Разделение на обучающую и тестовую выборки
```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
```

### Библиотека машинного обучения
Используйте библиотеку `scikit-learn`.

### Модели
```python
from sklearn.linear_model import LinearRegression, Ridge, Lasso
```

### Кодирование и масштабирование
Для удобной обработки числовых и категориальных признаков используйте:
```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
```

### Метрики качества
```python
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
```

Во многих версиях `scikit-learn` RMSE удобно вычислять так:
```python
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
```

Такой вариант совместим с разными версиями библиотеки. Использование записи
```python
mean_squared_error(..., squared=False)
```
может не работать в некоторых окружениях.

### Полезная идея
Удобно построить один и тот же конвейер предобработки, а затем подставлять в него разные модели.


## Решение


In [ ]:
# Поместите Ваш код сюда


## Вопросы для обсуждения

1. Почему задача предсказания `charges` является задачей регрессии, а не классификации?
2. Почему для Ridge и Lasso особенно важно масштабирование числовых признаков?
3. В чём содержательная разница между L1- и L2-регуляризацией?
4. Почему значения коэффициентов нельзя сравнивать напрямую, если признаки имеют разный масштаб?
5. В каких случаях регуляризация улучшает качество модели на тестовой выборке?


## Дополнительные задания

### Задание 1
Подберите параметр `alpha` для Ridge и Lasso:
- попробуйте несколько значений;
- сравните метрики;
- выберите лучший вариант.

### Задание 2
Постройте график зависимости качества модели от `alpha`.

### Задание 3
Сравните коэффициенты `LinearRegression`, `Ridge` и `Lasso` на одном графике.

### Задание 4
Добавьте полиномиальные признаки для числовых переменных и проверьте, изменится ли качество модели.

### Задание 5
Попробуйте интерпретировать влияние признака `smoker` на величину медицинских расходов.
